# 02-1 · The feasible set F

| | |
|---|---|
| **What it does** | drops every restaurant where ANY member has nothing safe to eat, before any scoring happens |
| **Proves** | choosing only from F makes a safety violation STRUCTURALLY impossible — not a matter of prompt quality |
| **Source** | `nbio.py` |
| **Guardrail** | `dish_is_safe` — checked directly below, including its known failure mode |
| **Eval surface** | flip one member's allergy and watch the feasible set shrink |

No API key needed.


In [1]:
# Locate the repo root (nbio.py lives there) regardless of where this notebook sits.
import os, sys
_root = os.getcwd()
for _ in range(4):
    if os.path.isfile(os.path.join(_root, "nbio.py")):
        break
    _root = os.path.dirname(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)


In [2]:
import nbio, json
nbio.bootstrap()


repo     : /project/painting/project-research/AnacodicAI-Labs/code/production-repos/ai-in-food/research_fair-mediator
data     : data  (ingredient-flavor.csv: found, 5 menu files)


PosixPath('/project/painting/project-research/AnacodicAI-Labs/code/production-repos/ai-in-food/research_fair-mediator')

## The guardrail, read directly

```python
def dish_is_safe(dish_name, diet, allergies):
    text = (dish_name or "").lower()
    for allergen in allergies or []:
        if allergen and allergen.lower() in text:
            return False
    diet = (diet or "mix").lower()
    if diet in {"veg", "vegetarian"}:
        return not is_nonveg_text(text)
    if diet == "vegan":
        return not is_nonveg_text(text) and not any(w in text for w in NON_VEGAN_WORDS)
    return True
```
`nbio.py` (this repo). It checks the dish NAME against keyword lists — nothing more.


In [3]:
# Step 1: probe the guardrail on cases that matter
probes = [
    ("Paneer Butter Masala", "veg", []),
    ("Paneer Butter Masala", "vegan", []),
    ("Chicken Biryani", "veg", []),
    ("Chicken Biryani", "non-veg", []),
    ("Peanut Chikki", "mix", ["peanuts"]),
    ("Crispy Baby Corn", "veg", ["gluten"]),   # the failure mode
]
nbio.table([{"dish": d, "diet": diet, "allergies": str(a), "safe": nbio.dish_is_safe(d, diet, a)}
            for d, diet, a in probes], cols=["dish", "diet", "allergies", "safe"], width=24)


dish                  diet     allergies    safe 
-------------------------------------------------
Paneer Butter Masala  veg      []           True 
Paneer Butter Masala  vegan    []           False
Chicken Biryani       veg      []           False
Chicken Biryani       non-veg  []           True 
Peanut Chikki         mix      ['peanuts']  True 
Crispy Baby Corn      veg      ['gluten']   True 


> **The last row is the known weakness, demonstrated rather than described.**
> "Crispy Baby Corn" is battered and fried — almost certainly containing gluten —
> but the filter only rejects a dish whose NAME contains the allergen string, so
> it passes for a coeliac diner. This is why the real catalogs use ingredient-level
> matching rather than trusting dish names.


## Build the feasible set on real data


In [4]:
# Step 2: load the group (4 real members from the harvested fixture) and the catalog
personas = json.load(open(nbio.SMALL_PERSONAS))
group_names = ["Ravi", "Marco", "Omar", "Alice"]
members = []
for name in group_names:
    p = next(x for x in personas if x["display_name"] == name)
    members.append({"member_id": name.lower(), "display_name": name,
                     "diet": p["diet_type"], "allergies": p["allergies"],
                     "taste": p["taste_vector"]})
for m in members:
    print(f"   {m['display_name']:<8} diet={m['diet']:<8} allergies={m['allergies']}")


   Ravi     diet=veg      allergies=[]
   Marco    diet=vegan    allergies=['dairy']
   Omar     diet=mix      allergies=[]
   Alice    diet=non-veg  allergies=['peanuts']


In [5]:
# Step 3: build a tiny catalog from the real menus, then the feasible set
def iter_dish_names(node):
    if isinstance(node, dict):
        if ("item" in node or "name" in node) and "price" in node:
            yield (node.get("item") or node.get("name") or "").strip()
            return
        for v in node.values():
            yield from iter_dish_names(v)
    elif isinstance(node, list):
        for v in node:
            yield from iter_dish_names(v)

catalog = []
for f in sorted(nbio.SMALL_MENUS.glob("*.json")):
    payload = json.loads(f.read_text())
    dishes = [n for n in iter_dish_names(payload) if n]
    catalog.append({"option_id": f.stem, "name": f.stem, "menu": [{"name": d} for d in dishes]})

before = {"catalog": len(catalog)}
F = nbio.build_feasible_set(catalog, members)
after = {"catalog": len(catalog), "F": len(F)}
nbio.io(1, "build_feasible_set", before, after, reads=["catalog", "members(diet, allergies)"])

for o in F:
    counts = {m["display_name"][:7]: len(o["safe_dishes"][m["member_id"]]) for m in members}
    print(f"   KEEP {o['name'][:34]:<36} {counts}")


+- STEP 1  build_feasible_set                                       [py]
| IN      : catalog, members(diet, allergies)
| NEW     : F
| CHANGED : -
+-------------------------------------------------------------------------
   KEEP Arabian Corner Restaurant - Skylou   {'Ravi': 23, 'Marco': 15, 'Omar': 60, 'Alice': 60}
   KEEP Casa.cafe                            {'Ravi': 70, 'Marco': 44, 'Omar': 129, 'Alice': 129}
   KEEP Delicia Multi Cuisine Restaurant     {'Ravi': 94, 'Marco': 67, 'Omar': 155, 'Alice': 155}
   KEEP House of Chefs                       {'Ravi': 123, 'Marco': 88, 'Omar': 198, 'Alice': 198}
   KEEP Ministry of South Bar and Kitchen    {'Ravi': 20, 'Marco': 15, 'Omar': 32, 'Alice': 32}


## The property, checked rather than trusted

If F is correct, every member has at least one safe dish at every option inside
it, and no dish in any safe set fails a re-check. Assert it.


In [6]:
# Step 4: assert the safety property that the paper's Methods section claims
violations = []
for o in F:
    for m in members:
        safe = o["safe_dishes"][m["member_id"]]
        if not safe:
            violations.append((o["name"], m["display_name"], "empty safe set"))
        for d in safe:
            if not nbio.dish_is_safe(d["name"], m["diet"], m["allergies"]):
                violations.append((o["name"], m["display_name"], d["name"]))

print("safety violations inside F:", len(violations))
assert not violations, violations[:5]
print("PASS -- every member has >=1 safe dish at every option in F,")
print("        and no dish in any safe set fails a re-check.")


safety violations inside F: 0
PASS -- every member has >=1 safe dish at every option in F,
        and no dish in any safe set fails a re-check.


### Reading it

The assertion above is the paper's core structural claim, made checkable: **a
constraint violation cannot occur downstream of this cell**, because F was built
by removing every option that would allow one. Contrast this with **05-mediate**,
where the LLM chat path receives the same constraints as English prose instead
of a structural filter — and violates them.

Next: **03-aggregate** scores every member against every option remaining in F.


## Safety stress test: does the filter bite, and do the results hold, when diners are far more restricted?

The Food.com diners carry synthetic diets and allergies at approximate US adult
rates, so few of them are restricted and the feasible set rarely shrinks. This
section compares two settings on Uber Eats Houston and Seattle (200 seeded groups each):

- **realistic**: the Food.com diners as built;
- **stress**: every diner gets 1-3 of the nine FDA major allergens, 20% are vegetarian and 10% vegan (seeded).

For each city and setting it reports the restaurants left per group, the groups
that lose any or all, one-shot vs mediator vs veto agreement, and the C1 result
at lambda 5 (`nbio.c1_evaluate`, with 2,000 bootstrap resamples). It also
reports the share of a menu that is unsafe for typical restricted profiles, over
60 seeded Houston restaurants.

Writes `figdata/ubereats-foodcom/safety_stress.csv` and `safety_dish_share.csv`.
The loader swap below replaces `nbio.load_personas` for the duration of the loop
and restores it afterwards. Slow: up to about an hour.

In [7]:
import csv, random, statistics as st

FDA = ["shellfish", "milk", "peanuts", "tree_nuts", "fish", "egg", "wheat", "soy", "sesame"]
STRESS_SEED = 7


def stressed(base):
    r = random.Random(STRESS_SEED)
    out = []
    for p in base:
        d = r.random()
        if d < 0.10:
            diet = "vegan"
        elif d < 0.30:
            diet = "vegetarian"
        else:
            diet = "non-veg"
        q = dict(p)
        q["diet_type"] = diet
        q["allergies"] = r.sample(FDA, r.randint(1, 3))
        out.append(q)
    return out


def write_stress(name, rows):
    out = nbio.HERE / "figdata" / "ubereats-foodcom" / name
    with open(out, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0]), lineterminator="\n")
        w.writeheader()
        w.writerows(rows)
    print("->", out.relative_to(nbio.HERE))

In [8]:
os.environ["FM_PERSONAS"] = "foodcom"
os.environ["FM_DATASET"] = "ubereats"
base = nbio.load_personas()
real_loader = nbio.load_personas
real_n_boot = nbio.C1_N_BOOT
stress_rows = []
try:
    for setting, loader in (("realistic", lambda: base), ("stress", lambda: stressed(base))):
        nbio.load_personas = loader
        for city in ("Houston, TX", "Seattle, WA"):
            os.environ["FM_CITY"] = city
            cat = nbio.load_catalog()
            groups = nbio.generate_groups(200, seed=STRESS_SEED)
            F = [len(nbio.build_feasible_set(cat, g["members"])) for g in groups]
            one = med = veto = n = 0
            for g in groups:
                r = nbio.run_group(g, cat, "fair", 0.5)
                if not r:
                    continue
                S, ids = r["_S"], r["_member_ids"]
                bar = nbio.consensus_bar(S, ids)
                n += 1
                one += min(S[r["pick_id"]][m] for m in ids) >= bar
                med += nbio.mediate(S, ids)["converged"]
                veto += nbio.mediate(S, ids, use_veto=True)["converged"]
            nbio.C1_N_BOOT = 2000
            res, _ = nbio.c1_evaluate(city, "ubereats", city)
            x = [y for y in res if y["formula"] == "whole menu" and y["lambda"] == 5][0]
            stress_rows.append({"setting": setting, "city": city, "n_groups": len(groups),
                                "F_mean": round(st.mean(F), 1), "F_min": min(F),
                                "groups_losing_any": sum(1 for f in F if f < len(cat)), "groups_empty": F.count(0),
                                "one_pick_agree": one, "mediator_agree": med, "veto_agree": veto,
                                "fair_pct_of_achievable": x["pct_of_achievable"], "fair_minsat_gain": x["minsat_gain"]})
            print(stress_rows[-1], flush=True)
finally:
    nbio.load_personas = real_loader
    nbio.C1_N_BOOT = real_n_boot
write_stress("safety_stress.csv", stress_rows)

{'setting': 'realistic', 'city': 'Houston, TX', 'n_groups': 200, 'F_mean': 298.8, 'F_min': 291, 'groups_losing_any': 26, 'groups_empty': 0, 'one_pick_agree': 98, 'mediator_agree': 180, 'veto_agree': 187, 'fair_pct_of_achievable': 67.1, 'fair_minsat_gain': 0.0269}


{'setting': 'realistic', 'city': 'Seattle, WA', 'n_groups': 200, 'F_mean': 298.4, 'F_min': 289, 'groups_losing_any': 58, 'groups_empty': 0, 'one_pick_agree': 95, 'mediator_agree': 180, 'veto_agree': 190, 'fair_pct_of_achievable': 61.9, 'fair_minsat_gain': 0.0222}


{'setting': 'stress', 'city': 'Houston, TX', 'n_groups': 200, 'F_mean': 292.4, 'F_min': 277, 'groups_losing_any': 146, 'groups_empty': 0, 'one_pick_agree': 99, 'mediator_agree': 180, 'veto_agree': 187, 'fair_pct_of_achievable': 67.1, 'fair_minsat_gain': 0.0271}


{'setting': 'stress', 'city': 'Seattle, WA', 'n_groups': 200, 'F_mean': 290.4, 'F_min': 272, 'groups_losing_any': 198, 'groups_empty': 0, 'one_pick_agree': 94, 'mediator_agree': 180, 'veto_agree': 191, 'fair_pct_of_achievable': 63.1, 'fair_minsat_gain': 0.0226}


-> figdata/ubereats-foodcom/safety_stress.csv


In [9]:
# Share of a menu that is unsafe for typical restricted profiles (Houston, 60 restaurants)
os.environ["FM_CITY"] = "Houston, TX"
cat = nbio.load_catalog()
sample = random.Random(STRESS_SEED).sample(cat, 60)
prof = [("milk allergy", "non-veg", ["milk"]), ("wheat allergy", "non-veg", ["wheat"]),
        ("vegetarian", "vegetarian", []), ("vegan", "vegan", []),
        ("milk + egg + wheat allergies", "non-veg", ["milk", "egg", "wheat"]),
        ("vegan + wheat + soy allergies", "vegan", ["wheat", "soy"])]
share_rows = []
for label, diet, allergies in prof:
    shares = []
    for o in sample:
        safe = nbio.safe_dishes_for({"diet": diet, "allergies": allergies}, o["menu"])
        shares.append(1 - len(safe) / len(o["menu"]))
    share_rows.append({"profile": label, "city": "Houston, TX", "n_restaurants": len(sample),
                       "share_of_menu_unsafe": round(st.mean(shares), 3)})
restricted = [p for p in base if p["allergies"] or p["diet_type"] != "non-veg"]
who = random.Random(STRESS_SEED).sample(restricted, 150)
shares = []
for p in who:
    for o in sample:
        safe = nbio.safe_dishes_for({"diet": p["diet_type"], "allergies": p["allergies"]}, o["menu"])
        shares.append(1 - len(safe) / len(o["menu"]))
share_rows.append({"profile": "average restricted Food.com diner", "city": "Houston, TX",
                   "n_restaurants": len(sample), "share_of_menu_unsafe": round(st.mean(shares), 3)})
for r in share_rows:
    print(r)
write_stress("safety_dish_share.csv", share_rows)

{'profile': 'milk allergy', 'city': 'Houston, TX', 'n_restaurants': 60, 'share_of_menu_unsafe': 0.536}
{'profile': 'wheat allergy', 'city': 'Houston, TX', 'n_restaurants': 60, 'share_of_menu_unsafe': 0.497}
{'profile': 'vegetarian', 'city': 'Houston, TX', 'n_restaurants': 60, 'share_of_menu_unsafe': 0.607}
{'profile': 'vegan', 'city': 'Houston, TX', 'n_restaurants': 60, 'share_of_menu_unsafe': 0.818}
{'profile': 'milk + egg + wheat allergies', 'city': 'Houston, TX', 'n_restaurants': 60, 'share_of_menu_unsafe': 0.767}
{'profile': 'vegan + wheat + soy allergies', 'city': 'Houston, TX', 'n_restaurants': 60, 'share_of_menu_unsafe': 0.879}
{'profile': 'average restricted Food.com diner', 'city': 'Houston, TX', 'n_restaurants': 60, 'share_of_menu_unsafe': 0.409}
-> figdata/ubereats-foodcom/safety_dish_share.csv
